In [1]:
# packages
import duckdb
import pandas as pd


In [2]:
# load data
# updated this table to the merged topics table because I've clustered outlier articles
# need to join in the original articles df to get the clean_body text of articles.
con = duckdb.connect("../guardian_articles.duckdb")
df_topics = con.execute(
    """
    SELECT * from article_topics_merged 
    """
).df()

df_main = con.execute(
    """
    SELECT * FROM article_topics
    """
).df()

con.close()


In [3]:
df_main.columns.tolist()

['id',
 'webTitle',
 'webPublicationDate',
 'clean_body',
 'topic_id',
 'topic_prob',
 'topic_label']

In [4]:
df_topics.columns.tolist()

['id',
 'webTitle',
 'webPublicationDate',
 'clean_body',
 'topic_id',
 'topic_prob',
 'topic_label',
 'source']

In [22]:
df_topics.groupby("topic_id")["topic_label"].nunique().max()

np.int64(1)

In [32]:
# identify top 20 topics
ranked_all_topics = df_topics["topic_label"].value_counts().reset_index()

ranked_all_topics.columns = ["topic", "count"]

In [33]:
print(ranked_all_topics)

                                             topic  count
0        0_prime minister_minister_australian_news  13766
1                                 -1_uncategorised   7883
2       1_premier league_liverpool_arsenal_chelsea    926
3    2_olympic_olympics_gold medal_winter olympics    136
4                 3_insomnia_sleep_waking_sleeping    110
..                                             ...    ...
112                      116_turkey_kurdish_and_of      6
113          110_bladder_urine_pelvic_incontinence      5
114                106_silica_dust_stone_silicosis      5
115              97_temperatures_snow_weather_cold      5
116                     99_venice_mose_city_lagoon      5

[117 rows x 2 columns]


In [ ]:
# label topics
# 4/13 update: will need to update these once I have new merged in topics from the outliers subcorpus
TOPIC_LABEL_OVERRIDES = {
    -1: "Uncategorized",
    0: "Government and Politics",
    1: "Football & Soccer",
    2: "Olympics",
    3: "Sleep",
    4: "Christmas Trees",
    5: "Classical Music",
    6: "Formula 1",
    7: "Basketball",
    8: "Cosmetic Products",
    9: "Space Exploration",
    10: "Poetry",
    11: "Middle East",
    12: "Catholicism and the Pope",
    13: "Fertility and Reproductive Medicine",
    14: "Nicotine and the Tobacco Industry",
    15: "Crosswords and Puzzles",
    16: "The Beatles",
    17: "Skiing",
    18: "Astronomy",
    19: "School Shootings & Gun Control",
    20: "Scents and Fragrances",
    21: "Brazilian Politics",
    22: "Video Games",
    23: "Hurricanes",
    24: "Pets",
    25: "King Charles",
    26: "Baldness",
    27: "Bees",
    28: "Pandas",
    29: "Air Pollution",
    30: "Cannabis",
    31: "World Chess",
    32: "Boxing and Mixed Martial Arts",
    33: "Nuclear War",
    34: "Language",
    35: "Danish Camping",
    36: "Mount Vesuvius",
    37: "History of the Holocaust",
    38: "Scooters",
    39: "Quarterback Tom Brady",
    40: "Earbud Headphones",
    41: "Brain Cognition",
    42: "Rivers",
    43: "Italian Politics",
    # 44: look into this one more,
    45: "Extinct Mammoths and Genetic Engineering",
    46: "James Bond",
    47: "Health Insurance",
    48: "Portugal",
    49: "Marine Life",
    50: "Gut Health",
    51: "Baseball",
    52: "Mayor of New York Zohran Mamdani",
    53: "Blindness",
    54: "Abortion Policy",
    # 55:  why is there no topic here?
    56: "Google Products",
    57: "Dinosaurs",
    # 58: look into these further
    59: "Glaciers",
    # 60: look into this further
    # 61: look into this further
    62: "Eye Health",
    63: "Bad Bunny's Super Bowl Halftime Performance",
    # 64: look into this furhter
    65: "Jeffery Epstein Connection to British Royal Family",
    66: "Dogs as Pets",
    67: "Ireland",
    # 68: look into this
    69: "Brain Diseases",
    70: "Eurovision",
    # 71: look into this
    # 72: look into this
    73: "Reproductive Health",
    # 74: look into this Doctor Who?
    75: "Elvis Presley",
    # 76: look into this
    # 77: maybe japan front in WW2
    # 78:
    # 79:
    80: "Electric Cars",
    81: "Solar Power",
    82: "Grenfell Tower Fire",
    # 83:
    # 84:
    85: "Traditional Chinese Medicine and Alternative Therapies",
    86: "Libraries and Books",
    # 87:
    # 88:
    # 89: "Alternative Energy in Memphis for XAI Data Centers",
    # 90:
    91: "Gang Crime",
    92: "Star Trek",
    # 93: "Black Holes",
    # 94:
    95: "Indonesia",
    96: "Diving",
    97: "Cold Temperatures",
    98: "Schools",
    # 99: "Venice",
    100: "Saving Money",
    101: "Major League Soccer",
    # 102:
    103: "Pac Man",
    # 104: "Maori"
    # 105: "Oscar Wilde",
    106: "Silicon Dust",
    107: "Ticket Pricing",
    108: "Paris",
    # 109:
    110: "Incontinence",
    # 111:
    # 112: "World Rugby",
    # 113:
    114: "South African Relations with the United States",
    115: "Medicare",
    116: "Turkey-Kurdish Relations",
}


In [ ]:
# create labelled table in DuckDB
df_topics["topic_label_clean"] = df_topics["topic_id"].map(
    lambda tid: TOPIC_LABEL_OVERRIDES.get(
        tid, topic_info.loc[tid, "Name"] if tid in topic_info.index else "Uncategorised"
    )
)

conn = duckdb.connect("../guardian_articles.duckdb")
conn.execute(
    "CREATE OR REPLACE TABLE article_topics_labelled AS SELECT * FROM df_topics"
)
conn.close()

print(f"Written {len(df_topics):,} rows to article_topics_labelled")
print(f"Overrides applied: {len(TOPIC_LABEL_OVERRIDES)} topics")
print(df_topics["topic_label_clean"].value_counts().head(20).to_string())


Written 24,660 rows to article_topics_labelled
Overrides applied: 19 topics
topic_label_clean
Government and Politics                13766
Uncategorised                           8705
Football & Soccer                        926
Olympics                                 136
Sleep                                    110
Trees                                     80
Classical Music                           47
Formula 1                                 36
Basketball                                34
Cosmetic Products                         34
Space Exploration                         32
Poetry                                    31
Middle East                               30
Catholicism and the Pope                  29
Nicotine and the Tobacco Industry         26
Fertility and Reproductive Medicine       26
Crosswords and Puzzles                    25
The Beatles                               25
Skiing                                    23
Astronomy                                 22
